# Experiment KPI Tracking

Track weekly KPI trends for SXO experiments defined in `data/experiments.csv`.

## What this notebook helps answer
- How are experiment pages trending week by week across key page-level KPIs?
- Which lines are the primary KPI focus for the current chart view?
- Where did each experiment start relative to the observed weekly series?

## How to use
1. Run **Setup (run once)**.
2. Run the remaining cells from top to bottom.
3. Use the KPI dropdown on the chart to switch between CTR, clicks, impressions, position, and engagement metrics.


In [ ]:
#@title Setup (run once)
import os
import sys
from datetime import date, timedelta
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    REPO_ROOT = Path("lla-data").resolve()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0"
else:
    REPO_ROOT = Path.cwd()
    if REPO_ROOT.name == "notebooks":
        REPO_ROOT = REPO_ROOT.parent
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import plotly.graph_objects as go
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import get_client, load_sql_template, run_query
from lla_data.url_utils import normalize_url_for_join

pd.set_option("display.max_colwidth", 160)
lifeline_theme.inject_fonts()
client = get_client()


In [ ]:
#@title Parameters and experiment input
EXPERIMENTS_CSV_PATH = REPO_ROOT / "data" / "experiments.csv"
PRE_PERIOD_WEEKS = 8

EXPECTED_COLUMNS = [
    "page-path",
    "focus-kpi",
    "experiment-start",
    "experiment-length-weeks",
    "kpi-start",
    "end-expected-kpi",
    "description",
]

KPI_CONFIG = {
    "ctr": {
        "column": "ctr",
        "label": "CTR",
        "yaxis_title": "CTR",
        "tickformat": ".2%",
        "hover_value": ".2%",
    },
    "clicks": {
        "column": "clicks",
        "label": "Clicks",
        "yaxis_title": "Clicks",
        "tickformat": ",.0f",
        "hover_value": ",.0f",
    },
    "impressions": {
        "column": "impressions",
        "label": "Impressions",
        "yaxis_title": "Impressions",
        "tickformat": ",.0f",
        "hover_value": ",.0f",
    },
    "avg_position": {
        "column": "avg_position",
        "label": "Average position",
        "yaxis_title": "Average position",
        "tickformat": ".2f",
        "hover_value": ".2f",
    },
    "organic_sessions": {
        "column": "organic_sessions",
        "label": "Organic sessions",
        "yaxis_title": "Organic sessions",
        "tickformat": ",.0f",
        "hover_value": ",.0f",
    },
    "engaged_sessions": {
        "column": "engaged_sessions",
        "label": "Engaged sessions",
        "yaxis_title": "Engaged sessions",
        "tickformat": ",.0f",
        "hover_value": ",.0f",
    },
    "page_views": {
        "column": "page_views",
        "label": "Page views",
        "yaxis_title": "Page views",
        "tickformat": ",.0f",
        "hover_value": ",.0f",
    },
}


def build_page_label(page_path: str) -> str:
    slug = page_path.strip("/").split("/")[-1]
    return slug.replace("-", " ").title()


experiments = pd.read_csv(EXPERIMENTS_CSV_PATH)
if list(experiments.columns) != EXPECTED_COLUMNS:
    raise ValueError(f"Expected CSV columns {EXPECTED_COLUMNS}, got {list(experiments.columns)}")

experiments["experiment-start"] = pd.to_datetime(experiments["experiment-start"])
experiments["experiment-length-weeks"] = experiments["experiment-length-weeks"].astype(int)
experiments["kpi-start"] = experiments["kpi-start"].astype(float)
experiments["end-expected-kpi"] = experiments["end-expected-kpi"].astype(float)
experiments["page-path"] = experiments["page-path"].map(normalize_url_for_join)
experiments["page-label"] = experiments["page-path"].map(build_page_label)

unsupported_kpis = sorted(set(experiments["focus-kpi"]) - set(KPI_CONFIG))
if unsupported_kpis:
    raise ValueError(f"Unsupported focus KPI values in CSV: {unsupported_kpis}")

selected_page_paths = experiments["page-path"].tolist()

print(f"Project: {config.PROJECT_ID}")
print(f"Search Console dataset: {config.SEARCHCONSOLE_DATASET}")
print(f"Experiments loaded: {len(experiments)}")
print(f"Focus KPIs present: {sorted(experiments['focus-kpi'].unique())}")
print("Canonical page paths:")
for page_path in experiments["page-path"]:
    print(f"- {page_path}")
experiments


In [ ]:
# Freshness check and chart window
freshness_sql = load_sql_template(
    "sql/notebooks/seo_search_freshness.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
    ga4_dataset=config.GA4_DATASET,
)

freshness = run_query(client, freshness_sql)
seo_max_date = pd.to_datetime(freshness.loc[0, "seo_max_date"]).date()
analysis_start_date = (
    experiments["experiment-start"].min() - pd.Timedelta(weeks=PRE_PERIOD_WEEKS)
).date()
max_planned_end_date = (
    experiments["experiment-start"]
    + pd.to_timedelta(experiments["experiment-length-weeks"] * 7 - 1, unit="D")
).max().date()
effective_end_date = min(seo_max_date, max_planned_end_date, date.today() - timedelta(days=2))

if analysis_start_date > effective_end_date:
    raise ValueError("Experiment analysis window is empty. Check CSV dates and data freshness.")

print(f"Latest SEO page date: {seo_max_date}")
print(f"Analysis window: {analysis_start_date} to {effective_end_date}")
print(f"Pre-period weeks: {PRE_PERIOD_WEEKS}")


In [ ]:
# Load weekly KPI series for all tracked experiment pages
page_sql = load_sql_template(
    "sql/notebooks/seo_experiment_weekly_kpis.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

page_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
    bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    bigquery.ArrayQueryParameter("selected_page_paths", "STRING", selected_page_paths),
]

weekly_metrics = run_query(client, page_sql, params=page_params)
if weekly_metrics.empty:
    raise ValueError("No weekly KPI data returned for the experiment pages.")

weekly_metrics["week_start"] = pd.to_datetime(weekly_metrics["week_start"])
weekly_metrics = weekly_metrics.sort_values(["page_order", "week_start"]).reset_index(drop=True)

chart_df = weekly_metrics.merge(
    experiments,
    left_on="page_path",
    right_on="page-path",
    how="left",
    validate="many_to_one",
)
chart_df["experiment-start-week"] = chart_df["experiment-start"].dt.to_period("W-SUN").dt.start_time

metric_columns = [cfg["column"] for cfg in KPI_CONFIG.values()]
long_df = chart_df.melt(
    id_vars=[
        "week_start",
        "page_path",
        "page_order",
        "page-label",
        "focus-kpi",
        "experiment-start",
        "experiment-start-week",
        "experiment-length-weeks",
        "kpi-start",
        "end-expected-kpi",
        "description",
    ],
    value_vars=metric_columns,
    var_name="kpi",
    value_name="value",
)
long_df["focus-match"] = long_df["kpi"] == long_df["focus-kpi"]

start_points = long_df[long_df["week_start"] == long_df["experiment-start-week"]].copy()

summary = (
    chart_df.groupby(["page-label", "focus-kpi"], as_index=False)
    .agg(
        weeks_returned=("week_start", "count"),
        clicks=("clicks", "sum"),
        impressions=("impressions", "sum"),
        organic_sessions=("organic_sessions", "sum"),
    )
)
summary


## KPI switcher chart

How to read this chart:
- Each line is one experiment page.
- The dropdown switches the KPI shown for all experiment pages.
- A full-opacity line means the selected KPI matches that page's `focus-kpi`.
- A 50% opacity line keeps the same page visible for context when the selected KPI is not its main focus.
- Diamond markers show the experiment start week on each series.


In [ ]:
# Interactive chart with KPI dropdown and experiment-start markers
page_order = experiments["page-path"].tolist()
page_labels = experiments.set_index("page-path")["page-label"].to_dict()
page_colors = {
    page_path: lifeline_theme.COLORWAY[idx % len(lifeline_theme.COLORWAY)]
    for idx, page_path in enumerate(page_order)
}

default_kpi = experiments["focus-kpi"].mode().iat[0]
date_start = long_df["week_start"].min()
date_end = long_df["week_start"].max()

fig = go.Figure()
trace_visibility = {}
trace_index = 0

for kpi_key, cfg in KPI_CONFIG.items():
    trace_visibility[kpi_key] = []
    for page_path in page_order:
        page_slice = long_df[(long_df["kpi"] == cfg["column"]) & (long_df["page_path"] == page_path)].copy()
        marker_slice = start_points[(start_points["kpi"] == cfg["column"]) & (start_points["page_path"] == page_path)].copy()
        if page_slice.empty:
            continue

        focus_match = bool(page_slice["focus-match"].iloc[0])
        opacity = 1.0 if focus_match else 0.5
        line_width = 3 if focus_match else 2
        line_customdata = page_slice[[
            "page_path",
            "focus-kpi",
            "experiment-start",
            "kpi-start",
            "end-expected-kpi",
            "description",
        ]].to_numpy()
        marker_customdata = marker_slice[[
            "page_path",
            "focus-kpi",
            "experiment-start",
            "kpi-start",
            "end-expected-kpi",
            "description",
        ]].to_numpy()

        fig.add_trace(
            go.Scatter(
                x=page_slice["week_start"],
                y=page_slice["value"],
                mode="lines+markers",
                name=page_labels[page_path],
                legendgroup=page_path,
                showlegend=True,
                visible=(kpi_key == default_kpi),
                line=dict(color=page_colors[page_path], width=line_width),
                marker=dict(color=page_colors[page_path], size=7),
                opacity=opacity,
                customdata=line_customdata,
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Week: %{x|%d %b %Y}<br>"
                    f"{cfg['label']}: %{{y:{cfg['hover_value']}}}<br>"
                    "Focus KPI: %{customdata[1]}<br>"
                    "Experiment start: %{customdata[2]|%d %b %Y}<br>"
                    "Planned KPI start: %{customdata[3]:.2%}<br>"
                    "Planned KPI end: %{customdata[4]:.2%}<br>"
                    "Page path: %{customdata[0]}<br>"
                    "%{customdata[5]}<extra></extra>"
                ),
            )
        )
        trace_visibility[kpi_key].append(trace_index)
        trace_index += 1

        fig.add_trace(
            go.Scatter(
                x=marker_slice["week_start"],
                y=marker_slice["value"],
                mode="markers",
                name=f"{page_labels[page_path]} start",
                legendgroup=page_path,
                showlegend=False,
                visible=(kpi_key == default_kpi),
                marker=dict(
                    color=page_colors[page_path],
                    size=14,
                    symbol="diamond",
                    line=dict(color="#ffffff", width=1),
                ),
                opacity=opacity,
                customdata=marker_customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Experiment start week: %{x|%d %b %Y}<br>"
                    f"Observed {cfg['label']}: %{{y:{cfg['hover_value']}}}<br>"
                    "Planned focus KPI start: %{customdata[3]:.2%}<br>"
                    "Planned focus KPI end: %{customdata[4]:.2%}<extra></extra>"
                ),
            )
        )
        trace_visibility[kpi_key].append(trace_index)
        trace_index += 1

for start_date in sorted(experiments["experiment-start"].dt.normalize().unique()):
    fig.add_vline(
        x=pd.Timestamp(start_date),
        line_dash="dot",
        line_color="#8080BB",
        opacity=0.6,
    )

buttons = []
for kpi_key, cfg in KPI_CONFIG.items():
    visible = [False] * len(fig.data)
    for idx in trace_visibility[kpi_key]:
        visible[idx] = True

    buttons.append(
        dict(
            label=cfg["label"],
            method="update",
            args=[
                {"visible": visible},
                {
                    "title": (
                        f"Weekly {cfg['label']} across experiment pages"
                        f"<br><span style='font-size:0.6em; font-weight:400;'>"
                        "Full opacity = selected KPI is that page's focus KPI"
                        "</span>"
                    ),
                    "yaxis": {
                        "title": cfg["yaxis_title"],
                        "tickformat": cfg["tickformat"],
                    },
                },
            ],
        )
    )

default_cfg = KPI_CONFIG[default_kpi]
fig.update_layout(
    template="lifeline",
    title=(
        f"Weekly {default_cfg['label']} across experiment pages"
        "<br><span style='font-size:0.6em; font-weight:400;'>"
        "Full opacity = selected KPI is that page's focus KPI"
        "</span>"
    ),
    xaxis_title="Week start",
    yaxis=dict(title=default_cfg["yaxis_title"], tickformat=default_cfg["tickformat"]),
    legend_title_text="Experiment page",
    hovermode="x unified",
    width=1600,
    height=900,
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1.0,
            xanchor="right",
            y=1.18,
            yanchor="top",
            buttons=buttons,
            showactive=True,
        )
    ],
)
fig.update_xaxes(range=[date_start, date_end])
lifeline_theme.add_lifeline_logo(fig)
fig.show()
